# L5 — Cross-Exchange Analysis

Charts X1–X3: cross-venue lag CDF, cumulative capacity, cascade throughput.

**Memory strategy**: free trades before OB load; compute capacity from trade rates only (no OB reload).

In [ ]:
import os, gc, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_trades
from mda.layer1.rates import compute_rate_timeseries, compute_rate_percentiles
from mda.layer5.cross_exchange import (
    compute_cross_venue_lag, compute_cumulative_capacity, detect_cascade_events,
)
from mda.plots.charts import (
    plot_cross_venue_lag_cdf, plot_cumulative_capacity,
    plot_cascade_throughput, save_figure,
)
print("imports OK")

## X1 — Cross-Venue Lag CDF

Sample 10% of BTC trades per exchange pair to limit memory.

In [ ]:
df = load_trades(DATA_DIR)
EXCHANGES = df["exchange"].unique().tolist()
print(f"Loaded {len(df):,} trades | exchanges: {EXCHANGES}")

In [ ]:
# Compute rate stats while df is loaded — needed for X2 and X3
rate_ts   = compute_rate_timeseries(df)
rate_pcts = compute_rate_percentiles(rate_ts)
print("Rate p99 per exchange:")
print(rate_pcts[["exchange","p99","p99_9"]].to_string(index=False))

In [ ]:
lags = compute_cross_venue_lag(df, symbol_filter="BTC", sample_frac=0.1)
print(f"Cross-venue lag pairs: {lags.groupby(['exchange_a','exchange_b']).size().to_dict()}")

fig = plot_cross_venue_lag_cdf(lags)
fig.show()
save_figure(fig, "X1_cross_venue_lag_cdf", REPORTS_DIR)
del lags
gc.collect()

## X2 — Cumulative Capacity

Trade-rate based only — avoids reloading 2.5 GB orderbook.

In [ ]:
# Build capacity from trade rates only (ob_update_rate=None)
capacity = compute_cumulative_capacity(rate_pcts, ob_update_rate=None)
print(capacity.to_string(index=False))

fig = plot_cumulative_capacity(capacity)
fig.show()
save_figure(fig, "X2_cumulative_capacity", REPORTS_DIR)
del capacity
gc.collect()

## X3 — Cascade Throughput

Free trades before cascade detection since df is still needed; reload it small.

In [ ]:
cascade_windows = detect_cascade_events(df, symbol_filter="BTC", pct_threshold=0.02)
print(f"Cascade events detected: {len(cascade_windows)}")

fig = plot_cascade_throughput(rate_ts, cascade_windows)
fig.show()
save_figure(fig, "X3_cascade_throughput", REPORTS_DIR)